# 01 · Exploración de Landing

Primera lectura de las ocho fuentes con PySpark y esquemas explícitos. Es la evidencia de exploración de la primera entrega: lo que el documento de diseño dice sobre los datos sale de acá.

Corre en Colab y en local. Las tablas quedan en `evidence/01_exploracion/`.

In [1]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Colab ya trae PySpark 4.0.4 y Java 21. Fijar la versión no instala nada si coincide
    # y evita sorpresas si Colab actualiza su imagen.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.4"], check=True)
    repo = Path("/content/cloud-provider-analytics")
    if not repo.exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/gonrc/cloud-provider-analytics.git", str(repo)], check=True)
    os.chdir(repo)
elif Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")
print("raíz del repo:", Path.cwd())

raíz del repo: /Users/grc/Documents/cloud-provider-analytics


In [2]:
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StructField, StructType

from cpa.casting import cast_with_fallback
from cpa.config import load_settings
from cpa.landing import extract, verify
from cpa.schemas import RESOURCES, SOURCES, USAGE_EVENTS, read_landing
from cpa.spark import get_spark

cfg = load_settings()
P = cfg["paths"]
EV = Path(P["evidence"]) / "01_exploracion"
EV.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

spark = get_spark(cfg)
print("Spark", spark.version, "| shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/04 21:01:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.4 | shuffle partitions: 8


## 1. Landing: extraer y verificar

Landing se extrae del zip de la cátedra, queda en solo lectura y se compara archivo por archivo contra `data/landing_manifest.csv` (SHA-256). Si alguien tocó un archivo, esta celda lo muestra.

In [3]:
n = extract(P["raw_zip"], P["landing"])
problems = verify(P["landing"], P["manifest"])
print("archivos extraídos en esta corrida:", n)
print("Landing intacta" if not problems else problems)

archivos extraídos en esta corrida: 0
Landing intacta


## 2. Inventario con esquema explícito

Una lectura por fuente con el esquema de `src/cpa/schemas.py`. `_corrupt_record` junta las filas que no respetan el esquema.

In [4]:
dfs = {name: read_landing(spark, P["landing"], name).cache() for name in SOURCES}

inv = []
for name, df in dfs.items():
    rel, fmt, _, key = SOURCES[name]
    n = df.count()
    inv.append({
        "fuente": name,
        "formato": fmt,
        "archivos": len(list((Path(P["landing"]) / rel).glob("*.jsonl"))) if fmt == "json" else 1,
        "filas": n,
        "columnas": len(df.columns) - 1,
        "filas_corruptas": df.filter("_corrupt_record is not null").count(),
        "clave_natural": "+".join(key),
        "duplicados_clave": n - df.select(*key).distinct().count(),
    })
inventario = pd.DataFrame(inv)
inventario.to_csv(EV / "inventario_fuentes.csv", index=False)
inventario

,fuente,formato,archivos,filas,columnas,filas_corruptas,clave_natural,duplicados_clave
0,customers_orgs,csv,1,80,11,0,org_id,0
1,users,csv,1,800,7,0,user_id,0
2,resources,csv,1,400,7,0,resource_id,0
3,support_tickets,csv,1,1000,8,0,ticket_id,0
4,marketing_touches,csv,1,1500,7,0,touch_id,0
5,nps_surveys,csv,1,92,4,0,org_id+survey_date,0
6,billing_monthly,csv,1,240,8,0,invoice_id,0
7,usage_events,json,120,43200,13,0,event_id,0


## 3. Nulos por columna

In [5]:
rows = []
for name, df in dfs.items():
    n = df.count()
    cols = [c for c in df.columns if c != "_corrupt_record"]
    nulls = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cols]).first().asDict()
    rows += [{"fuente": name, "columna": c, "nulos": v, "pct": round(100 * v / n, 1)} for c, v in nulls.items() if v]
nulos = pd.DataFrame(rows)
nulos.to_csv(EV / "nulos_por_columna.csv", index=False)
nulos

,fuente,columna,nulos,pct
0,customers_orgs,nps_score,11,13.8
1,users,last_login,139,17.4
2,resources,tags_json,83,20.8
3,support_tickets,resolved_at,240,24.0
4,support_tickets,csat,254,25.4
5,nps_surveys,nps_score,19,20.7
6,nps_surveys,comment,10,10.9
7,billing_monthly,credits,137,57.1
8,usage_events,value,877,2.0
9,usage_events,unit,2075,4.8


`carbon_kg` y `genai_tokens` no son nulos por mala calidad: no existen en v1, y `genai_tokens` solo aplica al servicio genai. Se ve en la sección 6.

## 4. Trampa: `value` llega a veces como texto

El README del dataset avisa que `value` puede venir como string. La pregunta es qué pasa si igual se lo declara double.

In [6]:
naive_schema = StructType([StructField(f.name, DoubleType()) if f.name == "value" else f for f in USAGE_EVENTS.fields])
naive = (spark.read.schema(naive_schema)
         .option("columnNameOfCorruptRecord", "_corrupt_record")
         .json(f"{P['landing']}/usage_events_stream/*.jsonl")
         .cache())

ev = cast_with_fallback(dfs["usage_events"], "value", "double").cache()

pd.DataFrame([
    {"lectura": "value declarado double", "value_nulo": naive.filter("value is null").count(),
     "filas_corruptas": naive.filter("_corrupt_record is not null").count()},
    {"lectura": "value como texto + cast", "value_nulo": ev.filter("value is null").count(),
     "filas_corruptas": ev.filter("_corrupt_record is not null").count(),
     "cast_fallidos": ev.filter("value_cast_failed").count()},
])

,lectura,value_nulo,filas_corruptas,cast_fallidos
0,value declarado double,2186,1309,NaN
1,value como texto + cast,877,0,0.0


In [7]:
naive.filter("_corrupt_record is not null").select("event_id", "value", "_corrupt_record").show(2, truncate=90)

+----------------+-----+------------------------------------------------------------------------------------------+
|        event_id|value|                                                                           _corrupt_record|
+----------------+-----+------------------------------------------------------------------------------------------+
|evt_gjhk88zfr7cm| NULL|{"event_id": "evt_gjhk88zfr7cm", "timestamp": "2025-08-31T20:24:00Z", "org_id": "org_ph...|
|evt_4623s647dmuz| NULL|{"event_id": "evt_4623s647dmuz", "timestamp": "2025-08-19T05:04:00Z", "org_id": "org_sg...|
+----------------+-----+------------------------------------------------------------------------------------------+
only showing top 2 rows


Con `value` declarado double, Spark marca como corruptas las 1.309 filas donde el número viene entre comillas y deja `value` nulo en 2.186 filas, cuando los nulos reales son 877. Leído como texto y casteado después no se pierde ninguna.

## 5. Trampa: comillas duplicadas en `resources.csv`

`tags_json` es una lista JSON dentro de un CSV, con las comillas escapadas duplicándolas (`"[""env:prod""]"`). El escape por defecto de Spark es la barra invertida.

In [8]:
sin_escape = (spark.read.schema(RESOURCES).option("header", True)
              .option("columnNameOfCorruptRecord", "_corrupt_record")
              .csv(f"{P['landing']}/resources.csv").cache())
print("sin escape='\"':", sin_escape.filter("_corrupt_record is not null").count(), "filas corruptas de", sin_escape.count())
print("con escape='\"':", dfs["resources"].filter("_corrupt_record is not null").count(), "filas corruptas")
dfs["resources"].select("resource_id", "service", "tags_json").show(4, truncate=False)

sin escape='"': 211 filas corruptas de 400
con escape='"': 0 filas corruptas
+------------+----------+------------------------+
|resource_id |service   |tags_json               |
+------------+----------+------------------------+
|res_eubfn9kr|compute   |["env:prod"]            |
|res_fvb66h3r|database  |NULL                    |
|res_cbrlqmn4|storage   |["env:prod", "pii:true"]|
|res_ew1yf0dw|networking|["pii:true"]            |
+------------+----------+------------------------+
only showing top 4 rows


## 6. Evolución de esquema: v1 y v2

In [9]:
ev.groupBy("schema_version").agg(
    F.min("timestamp").alias("desde"),
    F.max("timestamp").alias("hasta"),
    F.count("*").alias("eventos"),
    F.sum(F.col("carbon_kg").isNotNull().cast("int")).alias("con_carbon_kg"),
    F.sum(F.col("genai_tokens").isNotNull().cast("int")).alias("con_genai_tokens"),
).orderBy("schema_version").toPandas()

,schema_version,desde,hasta,eventos,con_carbon_kg,con_genai_tokens
0,1,2025-07-03 00:02:00,2025-07-17 23:56:00,10800,0,0
1,2,2025-07-18 00:01:00,2025-08-31 23:58:00,32400,32400,3132


In [10]:
(ev.filter("schema_version = 2")
   .groupBy("service")
   .agg(F.count("*").alias("eventos_v2"), F.sum(F.col("genai_tokens").isNotNull().cast("int")).alias("con_genai_tokens"))
   .orderBy("service").toPandas())

,service,eventos_v2,con_genai_tokens
0,analytics,3450,0
1,compute,9369,0
2,database,5559,0
3,genai,3132,3132
4,networking,5217,0
5,storage,5673,0


El corte es limpio: todo lo anterior al 18/7/2025 es v1 y todo lo posterior es v2. En v2 todos los eventos traen `carbon_kg` y solo los de genai traen `genai_tokens`.

## 7. Calidad: qué reglas van a hacer falta

Conteos sobre los datos crudos. No se corrige nada todavía: esto define las reglas de promoción a Silver.

In [11]:
b, t, u, c, m, r = (dfs[k] for k in ["billing_monthly", "support_tickets", "users", "customers_orgs", "marketing_touches", "resources"])

checks = [
    ("usage_events", "cost_usd_increment < -0.01", ev.filter("cost_usd_increment < -0.01").count()),
    ("usage_events", "value nulo", ev.filter("value is null").count()),
    ("usage_events", "unit nulo con value presente", ev.filter("unit is null and value is not null").count()),
    ("billing_monthly", "subtotal negativo", b.filter("subtotal < 0").count()),
    ("billing_monthly", "credits nulo", b.filter("credits is null").count()),
    ("billing_monthly", "USD con exchange_rate_to_usd distinto de 1", b.filter("currency = 'USD' and abs(exchange_rate_to_usd - 1) > 0.0001").count()),
    ("support_tickets", "csat fuera de 1 a 5", t.filter("csat < 1 or csat > 5").count()),
    ("support_tickets", "sin resolved_at (abiertos)", t.filter("resolved_at is null").count()),
    ("users", "last_login anterior a created_at", u.filter("last_login < created_at").count()),
    ("customers_orgs", "nps_score fuera de -100 a 100", c.filter("nps_score < -100 or nps_score > 100").count()),
    ("customers_orgs", "plan_tier e is_enterprise no coinciden", c.filter("(plan_tier = 'enterprise') != is_enterprise").count()),
    ("marketing_touches", "converted sin clicked", m.filter("converted and not clicked").count()),
    ("resources", "etiquetados pii:true", r.filter("tags_json like '%pii:true%'").count()),
    ("users", "email (dato personal)", u.filter("email is not null").count()),
]
orgs = c.select("org_id")
for name in ["usage_events", "users", "resources", "support_tickets", "billing_monthly", "nps_surveys", "marketing_touches"]:
    df = ev if name == "usage_events" else dfs[name]
    checks.append((name, "org_id que no existe en customers_orgs", df.join(orgs, "org_id", "left_anti").count()))
checks.append(("usage_events", "resource_id que no existe en resources", ev.join(r.select("resource_id"), "resource_id", "left_anti").count()))
checks.append(("usage_events", "service distinto al del recurso", ev.alias("e").join(r.alias("r"), "resource_id").filter("e.service != r.service").count()))
calidad = pd.DataFrame(checks, columns=["fuente", "hallazgo", "filas"])
calidad.to_csv(EV / "hallazgos_calidad.csv", index=False)
calidad

,fuente,hallazgo,filas
0,usage_events,cost_usd_increment < -0.01,211
1,usage_events,value nulo,877
2,usage_events,unit nulo con value presente,2038
3,billing_monthly,subtotal negativo,13
4,billing_monthly,credits nulo,137
5,billing_monthly,USD con exchange_rate_to_usd distinto de 1,160
6,support_tickets,csat fuera de 1 a 5,40
7,support_tickets,sin resolved_at (abiertos),240
8,users,last_login anterior a created_at,232
9,customers_orgs,nps_score fuera de -100 a 100,1


`unit` se puede completar sin adivinar: cada `metric` tiene una sola unidad (requests → count, cpu_hours → hours, storage_gb_hours → gb_hours).

In [12]:
ev.groupBy("metric").pivot("unit").count().fillna(0).toPandas()

,metric,null,count,gb_hours,hours
0,cpu_hours,505,0,0,10187
1,storage_gb_hours,654,0,12342,0
2,requests,916,18596,0,0


### Spikes de costo

Vista previa de tres criterios por servicio: z robusto (MAD) sobre el costo, el mismo z sobre `log(1 + costo)`, y costo mayor a 5 veces el p99. El umbral definitivo se fija en la segunda entrega.

In [13]:
def z_mad(df, col):
    # z robusto por servicio: 0.6745 * (x - mediana) / MAD
    med = df.groupBy("service").agg(F.percentile_approx(col, 0.5).alias("_med"))
    d = df.join(med, "service").withColumn("_dev", F.abs(F.col(col) - F.col("_med")))
    mad = d.groupBy("service").agg(F.percentile_approx("_dev", 0.5).alias("_mad"))
    return d.join(mad, "service").withColumn(f"z_{col}", 0.6745 * (F.col(col) - F.col("_med")) / F.col("_mad")).drop("_med", "_dev", "_mad")

pos = ev.filter("cost_usd_increment >= 0").withColumn("log_cost", F.log1p("cost_usd_increment"))
scored = z_mad(z_mad(pos, "cost_usd_increment"), "log_cost")
p99 = pos.groupBy("service").agg(F.percentile_approx("cost_usd_increment", 0.99).alias("p99"))
spikes = (scored.join(p99, "service")
            .groupBy("service")
            .agg(F.count("*").alias("eventos"),
                 F.round(F.percentile_approx("cost_usd_increment", 0.5), 3).alias("mediana"),
                 F.round(F.first("p99"), 3).alias("p99"),
                 F.round(F.max("cost_usd_increment"), 3).alias("maximo"),
                 F.sum((F.col("z_cost_usd_increment") > 3.5).cast("int")).alias("z_mad_costo"),
                 F.sum((F.col("z_log_cost") > 3.5).cast("int")).alias("z_mad_log"),
                 F.sum((F.col("cost_usd_increment") > 5 * F.col("p99")).cast("int")).alias("mayor_5x_p99"))
            .orderBy("service").toPandas())
spikes.to_csv(EV / "spikes_costo.csv", index=False)
spikes

,service,eventos,mediana,p99,maximo,z_mad_costo,z_mad_log,mayor_5x_p99
0,analytics,4571,1.232,11.626,249.386,1451,6,8
1,compute,12442,1.394,13.486,302.346,3922,5,26
2,database,7375,0.940,8.369,149.909,1863,11,11
3,genai,4137,2.077,19.727,317.431,1362,0,5
4,networking,6889,0.195,1.676,29.632,1573,16,12
5,storage,7570,0.360,3.340,62.992,2277,14,13


## 8. Trampa: los eventos no llegan en orden

Los 120 archivos simulan micro-lotes. Lo esperable sería que cada archivo cubra un tramo de tiempo; se mide qué rango de fechas trae cada uno.

In [14]:
# _metadata.file_name y no input_file_name(): la lectura es idéntica a la que ya está
# en caché, Spark la reemplaza por la versión en memoria e input_file_name() queda vacío.
arr = (read_landing(spark, P["landing"], "usage_events")
       .withColumn("archivo", F.regexp_extract(F.col("_metadata.file_name"), r"events_part_(\d+)", 1).cast("int"))
       .select("archivo", "timestamp")
       .cache())
rango = (arr.groupBy("archivo")
            .agg(F.min("timestamp").alias("primer_evento"), F.max("timestamp").alias("ultimo_evento"), F.count("*").alias("eventos"))
            .orderBy("archivo").toPandas())
rango["dias_cubiertos"] = ((rango.ultimo_evento - rango.primer_evento).dt.total_seconds() / 86400).round(1)
print(rango.dias_cubiertos.describe()[["min", "mean", "max"]].round(1).to_string())
rango.head()

min     59.0
mean    59.7
max     60.0


,archivo,primer_evento,ultimo_evento,eventos,dias_cubiertos
0,0,2025-07-03 00:08:00,2025-08-31 16:32:00,360,59.7
1,1,2025-07-03 02:38:00,2025-08-31 23:51:00,360,59.9
2,2,2025-07-03 14:16:00,2025-08-31 20:47:00,360,59.3
3,3,2025-07-03 00:48:00,2025-08-31 23:51:00,360,60.0
4,4,2025-07-03 10:43:00,2025-08-31 20:40:00,360,59.4


Cada archivo trae eventos de todo el período. Structured Streaming avanza el watermark al final de cada micro-lote (máximo event time visto menos el retraso) y los operadores con estado descartan lo que llega por debajo. La simulación siguiente reproduce ese cálculo con el orden real de llegada, para dos tamaños de micro-lote.

In [15]:
pdf = arr.withColumn("ts", F.unix_timestamp("timestamp")).select("archivo", "ts").toPandas()
delays = ["10min", "1h", "1D", "7D", "30D", "59D", "60D"]
rows = []
for k in [1, 10]:
    lote = pdf.archivo // k
    # máximo event time visto en los lotes anteriores (el primer lote no tiene watermark)
    max_previo = pdf.groupby(lote).ts.max().cummax().shift(1).reindex(lote).to_numpy()
    for d in delays:
        watermark = max_previo - pd.Timedelta(d).total_seconds()
        tardios = int((pdf.ts.to_numpy() < watermark).sum())
        rows.append({"max_files_per_trigger": k, "retraso": d, "descartados": tardios, "pct": round(100 * tardios / len(pdf), 1)})
sim = pd.DataFrame(rows)
sim.to_csv(EV / "simulacion_watermark.csv", index=False)
sim.pivot(index="retraso", columns="max_files_per_trigger", values="pct").reindex(delays)

max_files_per_trigger,1,10
retraso,,
10min,99.1,91.7
1h,99.1,91.6
1D,97.5,90.1
7D,87.6,81.0
30D,49.6,45.7
59D,1.6,1.5
60D,0.0,0.0


La simulación asume que los archivos entran en orden de nombre. Spark los toma por fecha de modificación, así que el número real puede variar un poco; la prueba siguiente lo mide con Structured Streaming.

### Prueba con Structured Streaming

Se procesa el directorio completo en micro-lotes de 10 archivos (`availableNow`), con deduplicación por `event_id` y tres configuraciones de watermark. La última corrida repite la tercera sobre el mismo checkpoint, para ver si vuelve a escribir algo.

In [16]:
import shutil
SPIKE = Path(P["checkpoints"]).parent / "_prueba_watermark"
shutil.rmtree(SPIKE, ignore_errors=True)
src = f"{P['landing']}/usage_events_stream"

def correr(nombre, watermark_col, retraso):
    stream = (spark.readStream.schema(USAGE_EVENTS)
              .option("maxFilesPerTrigger", cfg["streaming"]["max_files_per_trigger"])
              .json(src)
              .withColumn("ingest_ts", F.current_timestamp())
              .withWatermark(watermark_col, retraso)
              .dropDuplicatesWithinWatermark(["event_id"]))
    q = (stream.writeStream.format("parquet")
         .option("path", str(SPIKE / nombre / "out"))
         .option("checkpointLocation", str(SPIKE / nombre / "chk"))
         .trigger(availableNow=True).start())
    q.awaitTermination()
    ops = [p["stateOperators"][0] for p in q.recentProgress if p.get("stateOperators")]
    return {"configuracion": f"watermark {retraso} sobre {watermark_col}",
            "lotes": len(q.recentProgress),
            "filas_en_salida": spark.read.parquet(str(SPIKE / nombre / "out")).count(),
            "descartadas_por_watermark": sum(o.get("numRowsDroppedByWatermark", 0) for o in ops)}

prueba = pd.DataFrame([
    correr("event_ts_1h", "timestamp", "1 hour"),
    correr("event_ts_60d", "timestamp", "60 days"),
    correr("ingest_ts_1h", "ingest_ts", "1 hour"),
    {**correr("ingest_ts_1h", "ingest_ts", "1 hour"), "configuracion": "misma, re-ejecutada con el mismo checkpoint"},
])
prueba.to_csv(EV / "prueba_watermark_streaming.csv", index=False)
prueba

,configuracion,lotes,filas_en_salida,descartadas_por_watermark
0,watermark 1 hour sobre timestamp,12,7228,35972
1,watermark 60 days sobre timestamp,12,43200,0
2,watermark 1 hour sobre ingest_ts,13,43200,0
3,"misma, re-ejecutada con el mismo checkpoint",1,43200,0


### Cuántos archivos deja cada forma de particionar Bronze

Misma lectura en streaming, escrita particionando por la fecha del evento o por la fecha de ingesta.

In [17]:
def archivos_por_particion(nombre, particion):
    out = SPIKE / nombre
    stream = (spark.readStream.schema(USAGE_EVENTS)
              .option("maxFilesPerTrigger", cfg["streaming"]["max_files_per_trigger"])
              .json(src)
              .withColumn("ingest_ts", F.current_timestamp())
              .withColumn("event_date", F.to_date("timestamp"))
              .withColumn("ingest_date", F.to_date("ingest_ts")))
    q = (stream.writeStream.format("parquet").partitionBy(particion)
         .option("path", str(out / "out")).option("checkpointLocation", str(out / "chk"))
         .trigger(availableNow=True).start())
    q.awaitTermination()
    files = list((out / "out").rglob("*.parquet"))
    return {"particion": particion, "carpetas": len({f.parent for f in files}), "archivos": len(files),
            "eventos_por_archivo": round(43200 / len(files), 1),
            "kb_promedio": round(sum(f.stat().st_size for f in files) / len(files) / 1024, 1)}

archivos = pd.DataFrame([archivos_por_particion("bronze_por_event_date", "event_date"),
                         archivos_por_particion("bronze_por_ingest_date", "ingest_date")])
archivos.to_csv(EV / "archivos_por_particion.csv", index=False)
archivos

,particion,carpetas,archivos,eventos_por_archivo,kb_promedio
0,event_date,60,3600,12.0,5.6
1,ingest_date,1,60,720.0,40.0


Con una hora sobre el event time la salida pierde la mayor parte de los eventos. Con 60 días no se pierde nada, pero el estado de la deduplicación guarda todas las claves del período. Con el watermark sobre `ingest_ts` tampoco se pierde nada y el estado queda acotado a una hora de llegadas, que es la ventana en la que en la práctica llega un duplicado (el mismo archivo entregado dos veces). La re-ejecución con el mismo checkpoint no escribe filas nuevas. La decisión está en `DECISIONS.md` (D-06).

## 9. Distribución por organización y servicio

In [18]:
por_org = ev.groupBy("org_id").count().toPandas()["count"]
print(por_org.describe().round(1).to_string())
print("máximo / mediana:", round(por_org.max() / por_org.median(), 2))

count      80.0
mean      540.0
std       238.3
min        99.0
25%       384.8
50%       528.0
75%       657.8
max      1227.0
máximo / mediana: 2.32


In [19]:
(ev.groupBy("service")
   .agg(F.count("*").alias("eventos"), F.round(F.sum("cost_usd_increment"), 2).alias("costo_usd"))
   .orderBy(F.desc("eventos")).toPandas())

,service,eventos,costo_usd
0,compute,12498,61786.80
1,storage,7614,9407.14
2,database,7419,22635.46
3,networking,6921,4351.39
4,analytics,4590,19708.76
5,genai,4158,29544.43


## 10. Flujo batch de referencia: costo diario por organización y servicio

Es el mart `org_daily_usage_by_service` en su versión mínima. En términos de MapReduce: el filtro y la columna `usage_date` son el map, el `groupBy` por (org, día, servicio) es el shuffle y la agregación es el reduce. Spark además hace una agregación parcial antes del shuffle, que es el combiner de MapReduce. El join con `customers_orgs` va por broadcast porque la tabla tiene 80 filas.

In [20]:
cust = dfs["customers_orgs"].select("org_id", "plan_tier", "industry")
metric_sum = lambda name: F.sum(F.when(F.col("metric") == name, F.col("value"))).alias(name)

daily = (ev.filter("cost_usd_increment >= -0.01")
           .withColumn("usage_date", F.to_date("timestamp"))
           .groupBy("org_id", "usage_date", "service")
           .agg(F.round(F.sum("cost_usd_increment"), 4).alias("daily_cost_usd"),
                metric_sum("requests"), metric_sum("cpu_hours"), metric_sum("storage_gb_hours"),
                F.sum("genai_tokens").alias("genai_tokens"),
                F.round(F.sum("carbon_kg"), 6).alias("carbon_kg"),
                F.count("*").alias("eventos"))
           .join(F.broadcast(cust), "org_id", "left"))

import contextlib, io
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    daily.explain("formatted")
(EV / "plan_fisico_org_daily_usage.txt").write_text(buf.getvalue())
print("\n".join(l for l in buf.getvalue().splitlines()[:30] if l.strip()))

== Physical Plan ==
AdaptiveSparkPlan (21)
+- Project (20)
   +- BroadcastHashJoin LeftOuter BuildRight (19)
      :- HashAggregate (13)
      :  +- Exchange (12)
      :     +- HashAggregate (11)
      :        +- Project (10)
      :           +- Filter (9)
      :              +- InMemoryTableScan (1)
      :                    +- InMemoryRelation (2)
      :                          +- * Project (8)
      :                             +- * Project (7)
      :                                +- * Project (6)
      :                                   +- InMemoryTableScan (3)
      :                                         +- InMemoryRelation (4)
      :                                               +- Scan json  (5)
      +- BroadcastExchange (18)
         +- Filter (17)
            +- InMemoryTableScan (14)
                  +- InMemoryRelation (15)
                        +- Scan csv  (16)
(1) InMemoryTableScan
Output [8]: [carbon_kg#365, cost_usd_increment#363, genai_tokens#366L, m

In [21]:
daily = daily.cache()
entrada = ev.filter("cost_usd_increment >= -0.01").agg(F.sum("cost_usd_increment")).first()[0]
salida = daily.agg(F.sum("daily_cost_usd")).first()[0]
print("filas (org x día x servicio):", daily.count())
print(f"costo total entrada: {entrada:,.2f} USD | salida: {salida:,.2f} USD")
daily.orderBy("org_id", "usage_date", "service").limit(5).toPandas()

filas (org x día x servicio): 11050
costo total entrada: 148,351.58 USD | salida: 148,351.58 USD


,org_id,usage_date,service,daily_cost_usd,requests,cpu_hours,storage_gb_hours,genai_tokens,carbon_kg,eventos,plan_tier,industry
0,org_0lvsnujz,2025-07-03,analytics,8.8739,116.0,4.6168,12.7808,NaN,NaN,3,standard,Retail
1,org_0lvsnujz,2025-07-03,database,0.5723,NaN,1.7515,14.7891,NaN,NaN,3,standard,Retail
2,org_0lvsnujz,2025-07-03,storage,5.5542,255.0,NaN,4.6668,NaN,NaN,3,standard,Retail
3,org_0lvsnujz,2025-07-04,analytics,3.2915,NaN,2.7659,38.5678,NaN,NaN,3,standard,Retail
4,org_0lvsnujz,2025-07-04,database,12.7337,235.0,2.5139,41.2334,NaN,NaN,6,standard,Retail


El plan muestra `HashAggregate` parcial, `Exchange hashpartitioning` (el shuffle) y `HashAggregate` final, y el join sale como `BroadcastHashJoin`, sin shuffle. El plan completo quedó en `evidence/01_exploracion/plan_fisico_org_daily_usage.txt`.

## 11. Facturación: revenue en USD

Tres cosas de `billing_monthly` definen las reglas de D-10: el tipo de cambio de las facturas en USD, los subtotales negativos y el peso real de las facturas en ARS.

In [22]:
b = dfs["billing_monthly"]
multimoneda = b.groupBy("org_id").agg(F.countDistinct("currency").alias("monedas")).filter("monedas > 1")
fact = pd.DataFrame([
    ("facturas", b.count()),
    ("USD con exchange_rate_to_usd distinto de 1", b.filter("currency = 'USD' and abs(exchange_rate_to_usd - 1) > 0.0001").count()),
    ("subtotal negativo", b.filter("subtotal < 0").count()),
    ("subtotal negativo con impuesto positivo", b.filter("subtotal < 0 and taxes > 0").count()),
    ("impuesto distinto del 21% de |subtotal|", b.filter("abs(taxes - 0.21 * abs(subtotal)) > 0.011").count()),
    ("organizaciones con más de una moneda en los tres meses", multimoneda.count()),
], columns=["hallazgo", "valor"])
fact.to_csv(EV / "facturacion_hallazgos.csv", index=False)
fact

,hallazgo,valor
0,facturas,240
1,USD con exchange_rate_to_usd distinto de 1,160
2,subtotal negativo,13
3,subtotal negativo con impuesto positivo,13
4,impuesto distinto del 21% de |subtotal|,0
5,organizaciones con más de una moneda en los tr...,50


In [23]:
(b.groupBy("currency")
  .agg(F.count("*").alias("facturas"),
       F.round(F.percentile_approx("subtotal", 0.5), 2).alias("mediana_subtotal"),
       F.min("exchange_rate_to_usd").alias("tc_min"),
       F.max("exchange_rate_to_usd").alias("tc_max"))
  .orderBy("currency").toPandas())

,currency,facturas,mediana_subtotal,tc_min,tc_max
0,ARS,51,816.10,0.00133,0.00162
1,EUR,29,697.84,0.99810,1.19808
2,USD,160,654.61,0.85463,1.11791


En las 240 facturas el impuesto es el 21% del subtotal en valor absoluto, también en las 13 con subtotal negativo. Una nota de crédito tendría el impuesto en negativo, así que esas filas se toman como error y no entran al revenue. Un tipo de cambio de USD a USD solo puede ser 1.

El revenue de cada factura es (subtotal − créditos + impuestos) × tipo de cambio, con `credits` nulo igual a cero. La tabla compara el total con y sin las dos reglas.

In [24]:
fx = F.when(F.col("currency") == "USD", F.lit(1.0)).otherwise(F.col("exchange_rate_to_usd"))
fact_usd = (b.withColumn("total_moneda", F.col("subtotal") - F.coalesce("credits", F.lit(0.0)) + F.col("taxes"))
              .withColumn("revenue_tc_original", F.col("total_moneda") * F.col("exchange_rate_to_usd"))
              .withColumn("revenue_usd", F.col("total_moneda") * fx))
validas = fact_usd.filter("subtotal >= 0")
total = lambda df, col: round(df.agg(F.sum(col)).first()[0], 2)
revenue = pd.DataFrame([
    ("tipo de cambio como viene, con subtotales negativos", fact_usd.count(), total(fact_usd, "revenue_tc_original")),
    ("USD = 1, con subtotales negativos", fact_usd.count(), total(fact_usd, "revenue_usd")),
    ("USD = 1, sin subtotales negativos (D-10)", validas.count(), total(validas, "revenue_usd")),
], columns=["escenario", "facturas", "revenue_usd"])
revenue.to_csv(EV / "revenue_escenarios.csv", index=False)
revenue

,escenario,facturas,revenue_usd
0,"tipo de cambio como viene, con subtotales nega...",240,164184.90
1,"USD = 1, con subtotales negativos",240,164293.06
2,"USD = 1, sin subtotales negativos (D-10)",227,172252.22


In [25]:
(validas.groupBy("currency")
   .agg(F.count("*").alias("facturas"), F.round(F.sum("revenue_usd"), 2).alias("revenue_usd"))
   .orderBy("currency").toPandas())

,currency,facturas,revenue_usd
0,ARS,49,78.11
1,EUR,28,32829.39
2,USD,150,139344.72


Los montos en ARS son del mismo orden que los de USD, así que una factura en pesos queda en uno o dos dólares. Y la mayoría de las organizaciones cambia de moneda de un mes a otro. Se aplica la conversión como viene, que es lo que pide la consigna, y queda anotado en D-10. Un ejemplo:

In [26]:
ejemplo = (multimoneda.join(b.filter("currency = 'ARS'").select("org_id").distinct(), "org_id")
                      .orderBy("org_id").first()["org_id"])
(validas.filter(F.col("org_id") == ejemplo)
   .select("org_id", "month", "currency", "subtotal", "credits", "taxes", "exchange_rate_to_usd", F.round("revenue_usd", 2).alias("revenue_usd"))
   .orderBy("month").toPandas())

,org_id,month,currency,subtotal,credits,taxes,exchange_rate_to_usd,revenue_usd
0,org_0lvsnujz,2025-06-01,ARS,1316.02,NaN,276.36,0.00152,2.42
1,org_0lvsnujz,2025-07-01,USD,1600.22,21.0,336.05,1.05778,1915.27
2,org_0lvsnujz,2025-08-01,USD,675.07,NaN,141.77,1.00193,816.84


## 12. GenAI: tokens y costo estimado

La consulta 5 pide tokens y "costo estimado" por día. Se calculan dos costos: la suma de `cost_usd_increment` de los eventos de genai, y los tokens por el precio de referencia de `config/settings.toml` (D-12).

In [27]:
precio = cfg["genai"]["price_usd_per_million_tokens"]
g = (ev.filter("service = 'genai' and cost_usd_increment >= -0.01")
       .withColumn("usage_date", F.to_date("timestamp")))
con_tokens = g.filter("genai_tokens is not null")
tokens = con_tokens.agg(F.sum("genai_tokens")).first()[0]
costo_con_tokens = con_tokens.agg(F.sum("cost_usd_increment")).first()[0]
por_token = (con_tokens.filter("genai_tokens > 0")
               .select((F.col("cost_usd_increment") / F.col("genai_tokens")).alias("usd_por_token"))
               .agg(F.min("usd_por_token"), F.percentile_approx("usd_por_token", 0.5), F.max("usd_por_token")).first())
genai = pd.DataFrame([
    ("eventos de genai", g.count()),
    ("eventos con genai_tokens (v2)", con_tokens.count()),
    ("tokens", int(tokens)),
    ("costo de todos los eventos de genai (USD)", round(g.agg(F.sum("cost_usd_increment")).first()[0], 2)),
    ("costo de los eventos con tokens (USD)", round(costo_con_tokens, 2)),
    ("costo de eventos / tokens, USD por millón", round(costo_con_tokens / tokens * 1e6, 2)),
    ("USD por token por evento: mínimo", round(por_token[0], 4)),
    ("USD por token por evento: mediana", round(por_token[1], 4)),
    ("USD por token por evento: máximo", round(por_token[2], 4)),
    (f"tokens × {precio} USD por millón (USD)", round(tokens * precio / 1e6, 2)),
], columns=["medida", "valor"], dtype=object)
genai.to_csv(EV / "genai_resumen.csv", index=False)
genai

,medida,valor
0,eventos de genai,4137
1,eventos con genai_tokens (v2),3117
2,tokens,2545481
3,costo de todos los eventos de genai (USD),29710.44
4,costo de los eventos con tokens (USD),22263.35
5,"costo de eventos / tokens, USD por millón",8746.22
6,USD por token por evento: mínimo,0.0
7,USD por token por evento: mediana,0.0043
8,USD por token por evento: máximo,3.226
9,tokens × 2.0 USD por millón (USD),5.09


El costo de cada evento corresponde a su métrica (requests, horas de CPU o almacenamiento) y no a los tokens: dividido por los tokens va de 0 a más de 3 dólares por token. Por eso el mart publica los dos costos en columnas separadas.

In [28]:
genai_dia = (g.groupBy("usage_date")
               .agg(F.sum("genai_tokens").alias("genai_tokens"),
                    F.round(F.sum("cost_usd_increment"), 2).alias("costo_eventos_usd"))
               .withColumn("costo_tokens_usd", F.round(F.col("genai_tokens") * precio / 1e6, 4))
               .orderBy("usage_date").toPandas())
genai_dia.to_csv(EV / "genai_costo_por_dia.csv", index=False)
genai_dia.tail(5)

,usage_date,genai_tokens,costo_eventos_usd,costo_tokens_usd
55,2025-08-27,48346.0,452.22,0.0967
56,2025-08-28,49086.0,411.60,0.0982
57,2025-08-29,51700.0,456.30,0.1034
58,2025-08-30,61009.0,465.22,0.1220
59,2025-08-31,40339.0,299.09,0.0807


In [29]:
spark.stop()